In [1]:
import sys
#lets you access python's runtime environment
from pathlib import Path
#sys.path is a built in variable in the sys module and contains a list of directories that is seached through when you do an import
#so we are appending the src directory to that
sys.path.append(str(Path().resolve().parent / "src"))
import config

import pandas as pd
import datetime

In [2]:
f_cleaned = pd.read_csv(config.PROJECT_ROOT/ "data" / "cleaned_f.csv")
m_cleaned = pd.read_csv(config.PROJECT_ROOT/ "data" / "cleaned_m.csv")

f_cleaned['Date'] = pd.to_datetime(f_cleaned['Date'], format = '%Y-%m-%d')
m_cleaned['Date'] = pd.to_datetime(m_cleaned['Date'], format = '%Y-%m-%d')
f_sorted = f_cleaned.sort_values(['Name', 'Year', 'Date'])
m_sorted = m_cleaned.sort_values(['Name', 'Year', 'Date'])


C:\Users\bnpar\AppData\Local\Temp\ipykernel_25112\3413725149.py:1: DtypeWarning: Columns (33,38) have mixed types. Specify dtype option on import or set low_memory=False.
  f_cleaned = pd.read_csv(config.PROJECT_ROOT/ "data" / "cleaned_f.csv")
C:\Users\bnpar\AppData\Local\Temp\ipykernel_25112\3413725149.py:2: DtypeWarning: Columns (33,38) have mixed types. Specify dtype option on import or set low_memory=False.
  m_cleaned = pd.read_csv(config.PROJECT_ROOT/ "data" / "cleaned_m.csv")


Time from first competition to end of year in consideration

In [3]:
f_sorted['TimeCompeting'] = pd.to_datetime(f_sorted['Year'].astype(str) + '-12-31') - f_sorted.groupby('Name')['Date'].transform('min')
f_sorted['TimeCompeting'] = f_sorted['TimeCompeting'].dt.days


In [4]:
bomb_squat = (
    ((f_sorted['Squat1Kg'] <= 0) | f_sorted['Squat1Kg'].isna()) &
    ((f_sorted['Squat2Kg'] <= 0) | f_sorted['Squat2Kg'].isna()) &
    ((f_sorted['Squat3Kg'] <= 0) | f_sorted['Squat3Kg'].isna()) & 
    ((f_sorted['Best3SquatKg'] <= 0) | f_sorted['Best3SquatKg'].isna())
)

bomb_bench = (
    ((f_sorted['Bench1Kg'] <= 0) | f_sorted['Bench1Kg'].isna()) &
    ((f_sorted['Bench2Kg'] <= 0) | f_sorted['Bench2Kg'].isna()) &
    ((f_sorted['Bench3Kg'] <= 0) | f_sorted['Bench3Kg'].isna()) & 
    ((f_sorted['Best3BenchKg'] <= 0) | f_sorted['Best3BenchKg'].isna())
)

bomb_deadlift = (
    ((f_sorted['Deadlift1Kg'] <= 0) | f_sorted['Deadlift1Kg'].isna()) &
    ((f_sorted['Deadlift2Kg'] <= 0) | f_sorted['Deadlift2Kg'].isna()) &
    ((f_sorted['Deadlift3Kg'] <= 0) | f_sorted['Deadlift3Kg'].isna()) & 
    ((f_sorted['Best3DeadliftKg'] <= 0) | f_sorted['Best3DeadliftKg'].isna())
)


f_sorted['BombOut'] = bomb_squat | bomb_bench | bomb_deadlift

Number of times a lifter has bombed out. <br>
Number of meets since last bombout, capped at 999. If lifter has never bombed out this is set to the maximum cap of 999.

In [5]:
f_sorted['NumberOfBombOuts'] = f_sorted.groupby('Name')['BombOut'].cumsum()
f_sorted['MeetsSinceLastBombOut'] = f_sorted.groupby(['Name', 'NumberOfBombOuts']).cumcount()
f_sorted.loc[f_sorted['MeetsSinceLastBombOut']> config.CAP_MEETS_SINCE_BOMBOUT, 'MeetsSinceLastBombOut'] = config.CAP_MEETS_SINCE_BOMBOUT


#lifter has never bombed out so number of meets since last bombout is set to cap
f_sorted.loc[(f_sorted['NumberOfBombOuts'] ==0 ), 'MeetsSinceLastBombOut'] = config.CAP_MEETS_SINCE_BOMBOUT



In [6]:
attempt_cols = ['Squat1Kg','Squat2Kg','Squat3Kg',
                   'Bench1Kg','Bench2Kg','Bench3Kg',
                   'Deadlift1Kg','Deadlift2Kg','Deadlift3Kg']

f_sorted['AttemptsMade'] = (f_sorted[attempt_cols]>0).sum(axis=1)

In [7]:
f_sorted['LastMeetAttemptsMade'] = f_sorted.groupby(['Name', 'Year'])['AttemptsMade'].transform('last')
f_sorted['AverageAttemptsMade'] = f_sorted.groupby(['Name', 'Year'])['AttemptsMade'].transform('mean')
f_sorted['BestMeetOfYear'] = f_sorted.groupby(['Name', 'Year'])['TotalKg'].transform('max')


In [ ]:
# f_sorted['MeetsSinceLastBombOut'] = f_sorted.groupby(['Name', 'Year'])['MeetsSinceLastBombOut'].transform('last')
# f_sorted['NumberOfBombOuts'] = f_sorted.groupby(['Name', 'Year'])['NumberOfBombOuts'].transform('last')


In [9]:
panel_data = f_sorted.groupby(['Name', 'Year']).tail(1).reset_index(drop = True)


Index(['Name', 'Sex', 'Event', 'Equipment', 'Age', 'AgeClass',
       'BirthYearClass', 'Division', 'BodyweightKg', 'WeightClassKg',
       'Squat1Kg', 'Squat2Kg', 'Squat3Kg', 'Squat4Kg', 'Best3SquatKg',
       'Bench1Kg', 'Bench2Kg', 'Bench3Kg', 'Bench4Kg', 'Best3BenchKg',
       'Deadlift1Kg', 'Deadlift2Kg', 'Deadlift3Kg', 'Deadlift4Kg',
       'Best3DeadliftKg', 'TotalKg', 'Place', 'Dots', 'Wilks', 'Glossbrenner',
       'Goodlift', 'Tested', 'Country', 'State', 'Federation',
       'ParentFederation', 'Date', 'MeetCountry', 'MeetState', 'MeetTown',
       'MeetName', 'Sanctioned', 'Year', 'TimeCompeting', 'BombOut',
       'NumberOfBombOuts', 'MeetsSinceLastBombOut', 'AttemptsMade',
       'LastMeetAttemptsMade', 'AverageAttemptsMade', 'BestMeetTotal'],
      dtype='object')

In [8]:
###EDA vibes
null_attempt = f_sorted[
    f_sorted['Squat1Kg'].isna() |
    f_sorted['Squat2Kg'].isna() |
    f_sorted['Squat3Kg'].isna() |
    f_sorted['Bench1Kg'].isna() |
    f_sorted['Bench2Kg'].isna() |
    f_sorted['Bench3Kg'].isna() |
    f_sorted['Deadlift1Kg'].isna() |
    f_sorted['Deadlift2Kg'].isna() |
    f_sorted['Deadlift3Kg'].isna()
]

all_attempts_null = f_sorted[
    f_sorted['Squat1Kg'].isna() &
    f_sorted['Squat2Kg'].isna() &
    f_sorted['Squat3Kg'].isna() &
    f_sorted['Bench1Kg'].isna() &
    f_sorted['Bench2Kg'].isna() &
    f_sorted['Bench3Kg'].isna() &
    f_sorted['Deadlift1Kg'].isna() &
    f_sorted['Deadlift2Kg'].isna() &
    f_sorted['Deadlift3Kg'].isna()
]

print(len(null_attempt)/len(f_sorted))
print(len(all_attempts_null)/ len(f_sorted))

failed_attempt = f_sorted[
    (f_sorted['Squat1Kg'] < 0) |
    (f_sorted['Squat2Kg'] < 0) |
    (f_sorted['Squat3Kg'] < 0) |
    (f_sorted['Bench1Kg'] < 0) |
    (f_sorted['Bench2Kg'] < 0) |
    (f_sorted['Bench3Kg'] < 0) |
    (f_sorted['Deadlift1Kg'] < 0) |
    (f_sorted['Deadlift2Kg'] < 0) |
    (f_sorted['Deadlift3Kg'] < 0)
]


0.19136482817829548
0.16146715548646076
